# Phase 10.1: Needleman-Wunsch CUDA Pipeline Optimization


This notebook validates the Phase 10.1 CUDA pipeline optimization for Needleman-Wunsch global alignment. The phase keeps the existing wavefront recurrence and improves the execution pipeline with pinned host memory, optional cudaMallocAsync, reusable device buffers, batched execution, and CUDA streams.

In [ ]:
!nvidia-smi
!nvcc --version


In [ ]:
import os
print("Current working directory:", os.getcwd())


Compile the baseline and optimized GPU executables.

In [ ]:
!nvcc src/needleman_wunsch_gpu.cu \
  -O3 \
  -std=c++17 \
  -I src/common \
  -o needleman_wunsch_gpu

!nvcc src/needleman_wunsch_gpu_optimized.cu \
  -O3 \
  -std=c++17 \
  -I src/common \
  -o needleman_wunsch_gpu_optimized


Generate a fixed-length synthetic dataset for the optimized executable.

In [ ]:
!python scripts/generate_synthetic_dataset.py \
  --num-pairs 1000 \
  --sequence-length 64 \
  --output data/synthetic/synthetic_pairs_64_1000.txt \
  --seed 42


Run the optimized executable with batching and streams.

In [ ]:
!mkdir -p results/needleman_wunsch

!./needleman_wunsch_gpu_optimized \
  data/synthetic/synthetic_pairs_64_1000.txt \
  results/needleman_wunsch/needleman_wunsch_gpu_optimized_results.csv \
  --repetitions 5 \
  --batch-size 1024 \
  --num-streams 2 \
  --summary-only


Run the quick benchmark matrix and generate charts.

In [ ]:
!python benchmarks/run_needleman_wunsch_gpu_optimized_benchmark.py --quick


In [ ]:
!python scripts/plot_needleman_wunsch_gpu_optimized_benchmark.py


Display benchmark results.

In [ ]:
import pandas as pd
df = pd.read_csv("benchmarks/needleman_wunsch_gpu_optimized_benchmark_results.csv")
df


Display generated charts.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

chart_directory = Path("assets/benchmark_charts/needleman_wunsch_gpu_optimized")
for chart_path in sorted(chart_directory.glob("*.png")):
    print(chart_path)
    display(Image(filename=str(chart_path)))
